# NLP Capstone Project: Sentiment Analysis of Amazon Product Reviews

## Overview
This notebook builds a program that reads Amazon product reviews and predicts whether each one is **Positive**, **Negative** or **Neutral**. It then checks how good those predictions are by comparing them with the star ratings that customers gave.

**Tools used:** `pandas` (loading and cleaning data), `spaCy` with the `en_core_web_md` model (text processing), and `spacytextblob` (the sentiment score).

**Dataset:** *Datafiniti Amazon Consumer Reviews of Amazon Products* (May 2019) from Kaggle. The review text is in the `reviews.text` column and the star rating (1 to 5) is in `reviews.rating`.

## How the notebook is organised
Every function and every step has its own cell, so you can follow the work one piece at a time.

| Section | What it does |
|---|---|
| Setup | Imports, settings and loads the spaCy model |
| 1. Load and clean the data | Reads the CSV and removes rows with no review text |
| 2. Preprocessing and sentiment functions | Defines the functions that clean text and predict sentiment |
| 3. Demonstrations | Shows the cleaning, compares reviews and tests sample reviews |
| 4. Evaluation | Measures accuracy against the star ratings |
| 5. Results and what they mean | Explains what the results show |
| 6. Strengths, limitations and improvements | Summarises the conclusions |

## Before running
1. Install the libraries: `pip install pandas spacy spacytextblob`
2. Download the model and data: `python -m spacy download en_core_web_md` and `python -m textblob.download_corpora`
3. Keep the CSV file in the same folder as this notebook.
4. Use **Run All**, or run the cells from top to bottom. Later cells depend on earlier ones.

## Setup: libraries, settings and the spaCy model
The next three cells set everything up.

**Imports.** The `spacytextblob` import looks unused, but importing it is what registers the sentiment component with spaCy. The `# noqa: F401` comment stops code checkers from flagging it.

In [1]:
import pandas as pd
import spacy

# This import registers the "spacytextblob" pipeline component with spaCy.
# It is not used by name afterwards, so linters flag it as unused (F401).
from spacytextblob.spacytextblob import SpacyTextBlob  # noqa: F401

**Settings.** Keeping them together makes them easy to find and change:
- `DATA_FILE`: the name of the dataset file.
- `NEUTRAL_BAND`: reviews with a polarity score within this distance of zero (here +/-0.1) are called Neutral. The value was chosen by comparing several options in Section 4.
- `SAMPLE_SIZE` and `RANDOM_STATE`: the evaluation uses a random sample of 1,000 reviews. A fixed random state means the same sample is picked every time, so results can be repeated.
- `BANDS_TO_TEST` and `LABELS`: the neutral band values to compare, and the three sentiment labels.

In [2]:
# Name of the dataset file (kept in the same folder as this notebook)
DATA_FILE = "Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv"

# Polarity scores within +/- this value are treated as Neutral
# (0.1 chosen after comparing bands from 0.0 to 0.3 on a 1,000-review sample)
NEUTRAL_BAND = 0.1

# Settings for the evaluation against star ratings
SAMPLE_SIZE = 1000
RANDOM_STATE = 42  # makes the random sample repeatable
BANDS_TO_TEST = [0.0, 0.05, 0.1, 0.2, 0.3]
LABELS = ["Negative", "Neutral", "Positive"]

**Loading the model.** `en_core_web_md` is the medium English model. It includes word vectors, which are needed to compare how similar two reviews are. The `spacytextblob` component then adds a polarity score to every processed review.

In [3]:
# Load the medium English model (includes word vectors needed for similarity)
nlp = spacy.load("en_core_web_md")

# Add the TextBlob component so each Doc gets a polarity/sentiment score
nlp.add_pipe("spacytextblob")
print("Model loaded: en_core_web_md with the spacytextblob component added")

Model loaded: en_core_web_md with the spacytextblob component added


## 1. Load and clean the data
`load_reviews()` reads the CSV file into a pandas DataFrame, prints its size and column names, and counts missing values in `reviews.text`. It then removes any rows with no review text using `dropna(subset=['reviews.text'])`, because a review with no text cannot be analysed.

In [4]:
def load_reviews(file_path):
    """Load the reviews dataset and drop rows with no review text.

    Args:
        file_path: Path to the Datafiniti reviews CSV file.

    Returns:
        A tuple of (full dataframe, dataframe without missing review text).
    """
    dataframe = pd.read_csv(file_path)

    # Describe the data structure
    print("Dataset Shape (rows, columns):", dataframe.shape)
    print("\nColumn names:")
    print(dataframe.columns.tolist())
    print("\nMissing values in 'reviews.text':",
          dataframe["reviews.text"].isna().sum())

    # Remove rows where the review text is missing
    clean_data = dataframe.dropna(subset=["reviews.text"])
    print("Reviews remaining after dropping missing values:", len(clean_data))
    return dataframe, clean_data

**Run it.** The dataset has 28,332 rows and 24 columns and no missing review text, so no rows are removed here. The check is still included so the code works on other versions of the data.

In [5]:
# Load the dataset and keep the version with no missing review text
dataframe, clean_data = load_reviews(DATA_FILE)

Dataset Shape (rows, columns): (28332, 24)

Column names:
['id', 'dateAdded', 'dateUpdated', 'name', 'asins', 'brand', 'categories', 'primaryCategories', 'imageURLs', 'keys', 'manufacturer', 'manufacturerNumber', 'reviews.date', 'reviews.dateSeen', 'reviews.didPurchase', 'reviews.doRecommend', 'reviews.id', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.username', 'sourceURLs']

Missing values in 'reviews.text': 0
Reviews remaining after dropping missing values: 28332


**Select the review text.** `reset_index(drop=True)` renumbers the reviews from 0, so `reviews[0]`, `reviews[1]` and so on always work, even if rows had been dropped.

In [6]:
# Pull out just the review text, numbered from 0
reviews = clean_data["reviews.text"].reset_index(drop=True)
print("Reviews ready for analysis:", len(reviews))

Reviews ready for analysis: 28332


## 2. Preprocessing and sentiment functions
This section holds the main logic, split into small functions that each do one job.

| Function | Purpose |
|---|---|
| `preprocess_text` | Cleans a review: lowercases it, trims spaces, then removes stop words, punctuation and whitespace tokens |
| `label_from_polarity` | Turns a polarity score into Positive, Negative or Neutral using the neutral band |
| `predict_sentiment` | Cleans a review, scores it and returns the label and the polarity score |
| `predict_sentiment_raw` | Scores a review **without** cleaning, used only to compare with the cleaned version |
| `get_polarities` | Scores many reviews at once, which is faster than one at a time |
| `rating_to_label` | Converts a 1-5 star rating into a label (4-5 Positive, 3 Neutral, 1-2 Negative) |

**Key ideas**
- **Stop words** are very common words such as "the", "is" and "of" that carry little meaning. Removing them leaves the words that matter.
- **Polarity** is a score from -1 (very negative) to +1 (very positive), calculated by TextBlob from a dictionary of word scores.
- **A known risk:** spaCy treats words like "not", "no" and "less" as stop words. Removing them can change the meaning of a review. The later sections show real examples.

### `preprocess_text`
Cleans one review. It uses `nlp.make_doc`, which only splits text into tokens. That is all stop word removal needs, so it avoids running the whole spaCy pipeline and is much faster.

In [7]:
def preprocess_text(text):
    """Clean a review: lowercase, trim, and remove stop words and punctuation.

    Args:
        text: The raw review text.

    Returns:
        The cleaned review as a single string.
    """
    # str() guards against non-string values, lower()/strip() normalise text.
    # make_doc only tokenises, which is all is_stop/is_punct need, so it is
    # much faster than running the full spaCy pipeline.
    doc = nlp.make_doc(str(text).lower().strip())

    # Keep only meaningful tokens (no stop words, punctuation or whitespace)
    tokens = [
        token.text
        for token in doc
        if not token.is_stop and not token.is_punct and not token.is_space
    ]
    return " ".join(tokens)

### `label_from_polarity`
Converts a polarity score into a label. Anything within the neutral band of zero is Neutral.

In [8]:
def label_from_polarity(polarity, neutral_band):
    """Convert a polarity score to a sentiment label using a neutral band.

    Scores within +/- neutral_band are Neutral, above that Positive and
    below that Negative.

    Args:
        polarity: Polarity score from -1 to 1.
        neutral_band: Threshold for the Neutral classification.

    Returns:
        A sentiment label: "Positive", "Negative" or "Neutral".
    """
    if polarity > neutral_band:
        return "Positive"
    if polarity < -neutral_band:
        return "Negative"
    return "Neutral"

### `predict_sentiment`
The main function required by the task. It takes a raw review, cleans it, scores it with spaCyTextBlob and returns the label together with the polarity score.

In [9]:
def predict_sentiment(review):
    """Predict the sentiment of a product review.

    The review is cleaned first, then scored with spaCyTextBlob.
    A polarity above NEUTRAL_BAND is Positive, below -NEUTRAL_BAND is
    Negative, and anything in between is Neutral.

    Args:
        review: The raw review text.

    Returns:
        A tuple of (sentiment label, polarity score).
    """
    # Clean the review, then run it through the spaCy pipeline
    doc = nlp(preprocess_text(review))

    # Polarity ranges from -1 (very negative) to 1 (very positive)
    polarity = doc._.blob.polarity

    return label_from_polarity(polarity, NEUTRAL_BAND), polarity

### `predict_sentiment_raw`
Scores a review with no cleaning at all. It is only used to check whether stop word removal helps or hurts.

In [10]:
def predict_sentiment_raw(review):
    """Score a review without any preprocessing, for comparison.

    Args:
        review: The raw review text.

    Returns:
        The polarity score of the unprocessed review.
    """
    return nlp(str(review))._.blob.polarity

### `get_polarities`
Cleans and scores many reviews in one go using `nlp.pipe`, which processes text in batches and is faster than calling the model once per review.

In [11]:
def get_polarities(reviews):
    """Score many reviews at once (faster than one call per review).

    Args:
        reviews: An iterable of raw review texts.

    Returns:
        A list of polarity scores, one per review.
    """
    cleaned_reviews = [preprocess_text(review) for review in reviews]
    return [doc._.blob.polarity for doc in nlp.pipe(cleaned_reviews)]

### `rating_to_label`
Turns a customer's star rating into a sentiment label, so predictions can be compared with the ratings in Section 4.

In [12]:
def rating_to_label(rating):
    """Convert a numeric rating (1-5 stars) to a sentiment label.

    Args:
        rating: Numeric rating from 1 to 5.

    Returns:
        A sentiment label: "Positive", "Negative" or "Neutral".
    """
    if rating >= 4:
        return "Positive"
    if rating <= 2:
        return "Negative"
    return "Neutral"

## 3. Demonstrations: preprocessing, similarity and sample tests
These steps show the model working on small examples, which makes it easier to see what each function does.

**Note:** similarity measures how close two reviews are in *topic and vocabulary*, not whether they feel the same. Two reviews about batteries can score high even if one is praise and the other is a complaint.

### Check the text cleaning
`show_preprocessing_examples` prints the original and cleaned version of a few reviews, to check the cleaning works.

In [13]:
def show_preprocessing_examples(reviews, count=3):
    """Print raw and cleaned text for a few reviews to check the cleaning.

    Args:
        reviews: A Series of raw review texts.
        count: How many reviews to show.
    """
    for review in reviews[:count]:
        print("\nOriginal:", review)
        print("Cleaned: ", preprocess_text(review))

In [14]:
# Show the original and cleaned text for the first three reviews
show_preprocessing_examples(reviews)


Original: I order 3 of them and one of the item is bad quality. Is missing backup spring so I have to put a pcs of aluminum to make the battery work.
Cleaned:  order 3 item bad quality missing backup spring pcs aluminum battery work

Original: Bulk is always the less expensive way to go for products like these
Cleaned:  bulk expensive way products like

Original: Well they are not Duracell but for the price i am happy.
Cleaned:  duracell price happy


### Does cleaning help or hurt?
`compare_cleaned_and_raw` prints the polarity with and without stop word removal for the first five reviews.

In [15]:
def compare_cleaned_and_raw(reviews, count=5):
    """Print polarity with and without stop word removal.

    Args:
        reviews: A Series of raw review texts.
        count: How many reviews to compare.
    """
    for index in range(count):
        review = reviews[index]
        _, cleaned_polarity = predict_sentiment(review)
        raw_polarity = predict_sentiment_raw(review)
        print(f"Review {index}: cleaned = {cleaned_polarity:.3f}, "
              f"raw = {raw_polarity:.3f}")

In [16]:
# Compare polarity scores with and without stop word removal
compare_cleaned_and_raw(reviews)

Review 0: cleaned = -0.450, raw = -0.450
Review 1: cleaned = -0.500, raw = -0.333
Review 2: cleaned = 0.800, raw = 0.800
Review 3: cleaned = 0.500, raw = 0.500
Review 4: cleaned = 0.250, raw = 0.245


### Compare two reviews
`compare_reviews` uses spaCy's `similarity()` to compare two reviews. A score close to 1 means the reviews use very similar words and topics. It checks that the indexes are inside the dataset first, so it cannot go out of bounds.

In [17]:
def compare_reviews(reviews, index_one, index_two):
    """Print the similarity score between two reviews.

    Args:
        reviews: A Series of raw review texts.
        index_one: Position of the first review in reviews.
        index_two: Position of the second review in reviews.
    """
    # Guard against indexes outside the dataset (the task warns about this)
    last_index = len(reviews) - 1
    if not (0 <= index_one <= last_index and 0 <= index_two <= last_index):
        print(f"Indexes must be between 0 and {last_index}.")
        return

    review_one = reviews[index_one]
    review_two = reviews[index_two]

    # similarity() uses the word vectors in the en_core_web_md model
    similarity_score = nlp(review_one).similarity(nlp(review_two))

    print(f"\nReview {index_one}: {review_one}")
    print(f"Review {index_two}: {review_two}")
    print(f"Similarity score: {similarity_score:.3f}")

Two positive reviews about price:

In [18]:
compare_reviews(reviews, 3, 4)


Review 3: Seem to work as well as name brand batteries at a much better price
Review 4: These batteries are very long lasting the price is great.
Similarity score: 0.933


A negative review compared with a positive one:

In [19]:
compare_reviews(reviews, 0, 4)


Review 0: I order 3 of them and one of the item is bad quality. Is missing backup spring so I have to put a pcs of aluminum to make the battery work.
Review 4: These batteries are very long lasting the price is great.
Similarity score: 0.867


### Test on hand-written reviews
`test_sample_reviews` runs six reviews with a known expected sentiment through the model. This is a quick accuracy check on cases we control.

In [20]:
def test_sample_reviews():
    """Test the sentiment function on hand-written reviews of known sentiment."""
    test_reviews = [
        ("Great batteries, they last a long time and are very reliable.",
         "Positive"),
        ("The product stopped working after a week, very disappointed.",
         "Negative"),
        ("The product arrived on time and was as described, but I haven't "
         "used it enough to form an opinion.", "Neutral"),
        ("I love this product! It exceeded my expectations and I would "
         "definitely recommend it to others.", "Positive"),
        ("Terrible experience. The product was defective and customer "
         "service was unhelpful.", "Negative"),
        ("The product is okay, nothing special but does the job.", "Neutral"),
    ]

    for text, expected in test_reviews:
        label, polarity = predict_sentiment(text)
        raw_polarity = predict_sentiment_raw(text)
        print(f"\nReview: {text}")
        print(f"Expected sentiment: {expected} | "
              f"Predicted sentiment: {label}")
        print(f"Cleaned polarity score: {polarity:.3f} | "
              f"Raw polarity score: {raw_polarity:.3f}")

In [21]:
test_sample_reviews()


Review: Great batteries, they last a long time and are very reliable.
Expected sentiment: Positive | Predicted sentiment: Positive
Cleaned polarity score: 0.375 | Raw polarity score: 0.237

Review: The product stopped working after a week, very disappointed.
Expected sentiment: Negative | Predicted sentiment: Negative
Cleaned polarity score: -0.750 | Raw polarity score: -0.975

Review: The product arrived on time and was as described, but I haven't used it enough to form an opinion.
Expected sentiment: Neutral | Predicted sentiment: Neutral
Cleaned polarity score: 0.000 | Raw polarity score: 0.000

Review: I love this product! It exceeded my expectations and I would definitely recommend it to others.
Expected sentiment: Positive | Predicted sentiment: Positive
Cleaned polarity score: 0.250 | Raw polarity score: 0.312

Review: Terrible experience. The product was defective and customer service was unhelpful.
Expected sentiment: Negative | Predicted sentiment: Negative
Cleaned polarity 

## 4. Evaluation against the star ratings
Checking a handful of reviews by hand is not enough, so the model is also tested on 1,000 reviews chosen at random, using the customers' star ratings as the "right answers".

**Terms used**
- **Baseline:** the accuracy of a "model" that always answers Positive. Most reviews are positive, so this is a hard number to beat, and it shows whether the model is genuinely useful.
- **Confusion table:** rows are the actual labels and columns are the predicted ones, so it shows exactly which mistakes are made.
- **Recall:** of the reviews that really belong to a class, the share the model found.
- **Precision:** of the reviews the model put in a class, the share that really belong there.
- **Balanced recall:** the average recall of the three classes. Plain accuracy rewards leaning towards Positive, so this gives a fairer view of how the model treats each class.

### Rating distribution
First, how many reviews have each star rating. This shows how unbalanced the data is.

In [22]:
# Count how many reviews have each star rating (1 to 5)
print(clean_data["reviews.rating"].value_counts().sort_index())

reviews.rating
1      965
2      616
3     1206
4     5648
5    19897
Name: count, dtype: int64


### Build the scored sample
`build_scored_sample` picks the random sample and scores each review **once**. The scores are reused in all the later steps, which saves time.

In [23]:
def build_scored_sample(clean_data):
    """Take a repeatable random sample of reviews and score each one once.

    Args:
        clean_data: Dataframe of reviews with no missing review text.

    Returns:
        A dataframe with review text, rating, polarity and actual label.
    """
    sample = clean_data.dropna(subset=["reviews.rating"]).sample(
        n=SAMPLE_SIZE, random_state=RANDOM_STATE
    )
    sample = sample[["reviews.text", "reviews.rating"]].copy()

    # Score once, then reuse the polarity for every evaluation below
    sample["polarity"] = get_polarities(sample["reviews.text"])
    sample["actual_label"] = sample["reviews.rating"].apply(rating_to_label)
    return sample

In [24]:
# Take the random sample and score every review in it
sample = build_scored_sample(clean_data)
print("Reviews scored:", len(sample))

Reviews scored: 1000


### Accuracy, baseline and confusion table
`evaluate_on_sample` compares the predictions with the star-rating labels and prints the accuracy, the baseline and the confusion table.

In [25]:
def evaluate_on_sample(sample):
    """Print accuracy, baseline and confusion table for the sample.

    Args:
        sample: Scored sample from build_scored_sample().
    """
    predicted = sample["polarity"].apply(
        label_from_polarity, neutral_band=NEUTRAL_BAND
    )
    correct = (predicted == sample["actual_label"]).sum()
    accuracy = correct / len(sample) * 100

    # Baseline: accuracy from always predicting "Positive"
    baseline = (sample["actual_label"] == "Positive").mean() * 100

    print(f"Neutral band used: +/-{NEUTRAL_BAND}")
    print(f"Reviews evaluated: {len(sample)}")
    print(f"Correct predictions: {correct}")
    print(f"Accuracy against ratings: {accuracy:.1f}%")
    print(f"Baseline (always Positive): {baseline:.1f}%")

    # Rows are the actual labels, columns are the predicted labels
    print("\nConfusion table:")
    print(pd.crosstab(sample["actual_label"], predicted,
                      rownames=["Actual"], colnames=["Predicted"]))

In [26]:
evaluate_on_sample(sample)

Neutral band used: +/-0.1
Reviews evaluated: 1000
Correct predictions: 743
Accuracy against ratings: 74.3%
Baseline (always Positive): 90.0%

Confusion table:
Predicted  Negative  Neutral  Positive
Actual                                
Negative          8       17        25
Neutral           5       15        30
Positive         16      164       720


### Choosing the neutral band
`compare_neutral_bands` repeats the evaluation for several neutral band values and prints one table, so the options can be compared side by side.

In [27]:
def compare_neutral_bands(sample):
    """Compare accuracy and per-class recall for several neutral bands.

    Recall asks: of the reviews that truly are X, how many did we predict
    as X? Balanced recall averages the three classes, which is useful
    because the classes are very imbalanced.

    Args:
        sample: Scored sample from build_scored_sample().
    """
    rows = []
    for band in BANDS_TO_TEST:
        predicted = sample["polarity"].apply(
            label_from_polarity, neutral_band=band
        )
        accuracy = (predicted == sample["actual_label"]).mean() * 100

        recall = {}
        for label in LABELS:
            is_label = sample["actual_label"] == label
            recall[label] = (predicted[is_label] == label).mean() * 100

        rows.append({
            "Band": band,
            "Accuracy %": round(accuracy, 1),
            "Negative recall %": round(recall["Negative"]),
            "Neutral recall %": round(recall["Neutral"]),
            "Positive recall %": round(recall["Positive"]),
            "Balanced recall %": round(sum(recall.values()) / len(LABELS), 1),
        })

    # Print the table once, after every band has been evaluated
    print(pd.DataFrame(rows).to_string(index=False))

In [28]:
compare_neutral_bands(sample)

 Band  Accuracy %  Negative recall %  Neutral recall %  Positive recall %  Balanced recall %
 0.00        78.3                 24                12                 85               40.3
 0.05        76.7                 20                18                 83               40.4
 0.10        74.3                 16                30                 80               42.0
 0.20        69.1                 10                50                 73               44.5
 0.30        59.2                  4                64                 62               43.3


## 5. Results and what they mean
These tables come from running the notebook on the full dataset. If you change a setting such as `NEUTRAL_BAND` and re-run, update them to match.

### The dataset
28,332 reviews, 24 columns, no missing review text. The ratings are heavily skewed towards positive reviews:

| Rating | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| Reviews | 965 | 616 | 1,206 | 5,648 | 19,897 |

About 90.2% of reviews have 4 or 5 stars, 5.6% have 1 or 2 stars and 4.3% have 3 stars. This imbalance affects how the results should be read.

### Preprocessing examples
| Original | Cleaned |
|---|---|
| I order 3 of them and one of the item is bad quality. Is missing backup spring... | order 3 item bad quality missing backup spring pcs aluminum battery work |
| Bulk is always the less expensive way to go for products like these | bulk expensive way products like |
| Well they are not Duracell but for the price i am happy. | duracell price happy |

The cleaning works as intended, but the last two rows show the cost: "less expensive" became "expensive", and "not Duracell" lost its "not".

### Cleaned vs unprocessed polarity (first five reviews)
| Review | Cleaned | Unprocessed |
|---|---|---|
| 0 | -0.450 | -0.450 |
| 1 | -0.500 | -0.333 |
| 2 | 0.800 | 0.800 |
| 3 | 0.500 | 0.500 |
| 4 | 0.250 | 0.245 |

Stop word removal changed the score on two of the five reviews, but not enough to change any label. Review 1 ("less expensive") is actually a positive review but is scored negative both ways, so the main cause is the word dictionary, not the cleaning.

### Similarity
Reviews 3 and 4 (both positive, both about price) scored **0.933**. Reviews 0 and 4 (one negative, one positive) still scored **0.867**. This confirms that similarity measures topic and vocabulary, not sentiment.

### Hand-written test reviews
Five of the six were correct. The miss was *"The product is okay, nothing special but does the job."*, expected Neutral but predicted Positive (0.429). The dictionary scores "okay" and "special" as positive and does not understand "nothing special".

### Evaluation against star ratings
On the 1,000-review sample the model reached **74.3% accuracy** (743 correct), but the **baseline is 90.0%**: always answering Positive would have scored higher. Accuracy alone is therefore misleading here, and the per-class results matter more.

| Actual \ Predicted | Negative | Neutral | Positive | Recall |
|---|---|---|---|---|
| Negative (50) | 8 | 17 | 25 | 16% |
| Neutral (50) | 5 | 15 | 30 | 30% |
| Positive (900) | 16 | 164 | 720 | 80% |
| **Precision** | 28% | 8% | 93% | |

- When the model says Positive, it is usually right (93% precision).
- It misses most negative reviews: half of them (25 of 50) were called Positive.
- 164 positive reviews were labelled Neutral, so the Neutral class is very unreliable (8% precision).

### Choosing the neutral band
| Band | Accuracy | Negative recall | Neutral recall | Positive recall | Balanced recall |
|---|---|---|---|---|---|
| 0.0 | 78.3% | 24% | 12% | 85% | 40.3% |
| 0.05 | 76.7% | 20% | 18% | 83% | 40.4% |
| **0.1 (used)** | 74.3% | 16% | 30% | 80% | 42.0% |
| 0.2 | 69.1% | 10% | 50% | 73% | 44.5% |
| 0.3 | 59.2% | 4% | 64% | 62% | 43.3% |

A wider band finds more Neutral reviews but loses accuracy and Negative recall at every step. 0.1 was chosen as a compromise. Balanced recall is highest at 0.2, but that setting loses more accuracy and catches fewer negative reviews. The gains are modest, so the neutral band does not fix the model's main weaknesses.

## 6. Strengths, limitations and improvements

### Strengths
- Fast and simple: no training data or model training is needed, and the polarity score is easy to interpret.
- Works well on reviews with clear sentiment words such as "great", "terrible" or "happy".
- The code is modular, so each step (cleaning, scoring, labelling) can be changed on its own.
- The neutral band raised Neutral recall from 12% to 30%.

### Limitations
- **No understanding of context:** each word is scored on its own, so phrases such as "less expensive" and "nothing special" are misread.
- **Stop word removal can remove meaning:** words like "not", "no" and "less" are on spaCy's stop word list.
- **Weak on minority classes:** only 16% of negative and 30% of neutral reviews were found, and accuracy (74.3%) is below the 90.0% baseline.
- **Evaluation caveats:** the sample had only 50 negative and 50 neutral reviews, so those percentages are rough. The neutral band was chosen and tested on the same sample. Star ratings are an imperfect stand-in for sentiment, and treating 3 stars as Neutral is an assumption.
- **Similarity measures topic, not sentiment.**

### Possible improvements
- Keep negation words (such as "not") when cleaning, or score the unprocessed text.
- Train a classifier, for example logistic regression on TF-IDF features or a transformer model, using the star ratings as labels, with class weighting to handle the imbalance.
- Pick settings on one sample and confirm them on a separate held-out sample.